# MindNova Hybrid Risk Prediction Pipeline
## Behavioral scoring without diagnostic leakage

This notebook orchestrates the development of a production-safe mental health risk model. It uses diagnostic scores only for offline labeling and relies entirely on behavioral/lifestyle signals for inference.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Add src to path
sys.path.append(os.path.abspath('../src'))

import hybrid_preprocess as preprocess
import hybrid_feature_engineering as engineering
import hybrid_train as training
import hybrid_evaluate as evaluate
import hybrid_tune as tuning
import hybrid_explain as explain

## 1. Data Labeling & Preprocessing
Scaling PHQ9 + GAD7 to a 0-100 `RiskScore` and mapping to `RiskCategory` (Low, Moderate, High).

In [ ]:
PATH = '../data/raw/Univsersiyt_Student_Mental_health_data.csv'
df = preprocess.load_data(PATH)
df_labeled = preprocess.create_hybrid_labels(df)
preprocess.basic_inspection(df_labeled)

## 2. Hybrid Feature Engineering
Implementing 10 specialized behavioral metrics and ratios.

In [ ]:
df_feat = engineering.engineer_hybrid_features(df_labeled)
df_final = preprocess.drop_diagnostic_features(df_feat)

print(f"Final Predictors: {df_final.columns.tolist() if hasattr(df_final, 'columns') else 'N/A'}")

## 3. Model Training & Evaluation

In [ ]:
X = df_final.drop(columns=['RiskCategory'])
y = df_final['RiskCategory']

X_scaled, scaler = engineering.scale_features(X, X.columns)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
X_res, y_res = training.solve_multiclass_imbalance(X_train, y_train)

models = training.train_hybrid_models(X_res, y_res)

summary_metrics = []
for name, model in models.items():
    m, _, _ = evaluate.evaluate_multiclass_model(name, model, X_test, y_test)
    summary_metrics.append(m)

display(pd.DataFrame(summary_metrics))

## 4. Hyperparameter Tuning & Selection

In [ ]:
best_model, params = tuning.tune_hybrid_xgboost(X_res, y_res)
print(f"Best Params: {params}")

final_metrics, y_pred, y_prob = evaluate.evaluate_multiclass_model("Tuned Hybrid XGBoost", best_model, X_test, y_test)
evaluate.plot_multiclass_cm(y_test, y_pred, "Final Hybrid Model Confusion Matrix")

## 5. Model Explainability

In [ ]:
explain.generate_hybrid_shap_plots(best_model, X_res, X_test, X.columns)

drivers, protective = explain.get_risk_drivers(best_model, X.columns)
print(f"\n🔥 Top Risk Drivers: {drivers}")
print(f"🛡️ Top Protective Factors: {protective}")